# 05 — Leave-One-Facility-Out Peer Benchmark Calculation and Validation

## Purpose

This notebook calculates the governed descriptive peer benchmarks defined in
Notebook 02 and populates the benchmark fields reserved in Notebook 04.

The notebook implements:

- Leave-one-facility-out peer benchmarking
- Primary APR-DRG × severity peer groups
- APR-DRG fallback peer groups
- Minimum comparison-volume enforcement
- Separate LOS and estimated-cost eligibility rules
- Benchmark-level assignment
- Benchmark coverage diagnostics
- Direct mathematical reconciliation
- Preservation of the physical fact-table grain and schema
- Power BI-ready benchmarked fact-table export
- Machine-readable benchmark validation outputs
- Business-readable benchmark methodology documentation

## Inputs

- `outputs/metric_catalog/benchmark_specification.csv`
- `outputs/metric_catalog/validation_results.csv`
- `outputs/star_schema/column_specification.csv`
- `outputs/star_schema/schema_validation_results.csv`
- `outputs/physical_model/physical_validation_results.csv`
- `outputs/physical_model/parquet_validation.csv`
- `outputs/physical_model/tables/FactDischarge.parquet`
- `outputs/physical_model/tables/DimHospital.parquet`
- `outputs/physical_model/tables/DimService.parquet`
- `outputs/physical_model/tables/DimCaseMix.parquet`

## Outputs

- `outputs/peer_benchmarks/tables/FactDischarge.parquet`
- `outputs/peer_benchmarks/benchmark_standards.csv`
- `outputs/peer_benchmarks/benchmark_specification_snapshot.csv`
- `outputs/peer_benchmarks/benchmark_eligibility.csv`
- `outputs/peer_benchmarks/benchmark_coverage.csv`
- `outputs/peer_benchmarks/benchmark_peer_size_summary.csv`
- `outputs/peer_benchmarks/manual_reconciliation.csv`
- `outputs/peer_benchmarks/selection_logic_validation.csv`
- `outputs/peer_benchmarks/benchmark_validation_results.csv`
- `outputs/peer_benchmarks/parquet_validation.csv`
- `outputs/peer_benchmarks/export_manifest.csv`
- `docs/peer_benchmarks.md`

## Benchmark Design

Primary peer group: `APR-DRG Code × APR Severity of Illness Code`

Fallback peer group: `APR-DRG Code`

For every focal hospital, that hospital's eligible records are excluded from
its own comparison group. A benchmark is available only when at least 30
eligible comparison discharges remain after excluding the focal hospital.

LOS and estimated-cost benchmarks are calculated independently because their
valid-record populations may differ.

These are descriptive operational peer benchmarks. They are not machine-learning
predictions, formal clinical risk adjustment, causal estimates, quality ratings,
or audited financial expectations.

## 1. Imports

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

## 2. Project Paths

In [2]:
def find_project_root(start_path):
    """Find the repository root using the existing project charter."""

    for candidate in [start_path, *start_path.parents]:
        if (candidate / "docs" / "project_charter.md").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Expected docs/project_charter.md "
        "in the current directory or one of its parents."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

CATALOG_DIR = PROJECT_ROOT / "outputs" / "metric_catalog"
SCHEMA_DIR = PROJECT_ROOT / "outputs" / "star_schema"
PHYSICAL_DIR = PROJECT_ROOT / "outputs" / "physical_model"
PHYSICAL_TABLE_DIR = PHYSICAL_DIR / "tables"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "peer_benchmarks"
TABLE_DIR = OUTPUT_DIR / "tables"
WORK_DIR = OUTPUT_DIR / "_work"
DOCS_DIR = PROJECT_ROOT / "docs"

WORK_DB_PATH = WORK_DIR / "peer_benchmark_build.duckdb"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Physical model directory:", PHYSICAL_DIR)
print("Peer benchmark output directory:", OUTPUT_DIR)

Project root: C:\Users\kdy10\OneDrive\Documents\Python\Python ML Project\04_hospital_operations_powerbi
Physical model directory: C:\Users\kdy10\OneDrive\Documents\Python\Python ML Project\04_hospital_operations_powerbi\outputs\physical_model
Peer benchmark output directory: C:\Users\kdy10\OneDrive\Documents\Python\Python ML Project\04_hospital_operations_powerbi\outputs\peer_benchmarks


## 3. Load Upstream Specifications and Validation Outputs

In [3]:
input_files = {
    "benchmark_specification": CATALOG_DIR / "benchmark_specification.csv",
    "catalog_validation": CATALOG_DIR / "validation_results.csv",
    "column_specification": SCHEMA_DIR / "column_specification.csv",
    "schema_validation": SCHEMA_DIR / "schema_validation_results.csv",
    "physical_validation": PHYSICAL_DIR / "physical_validation_results.csv",
    "physical_parquet_validation": PHYSICAL_DIR / "parquet_validation.csv",
}

physical_table_paths = {
    "FactDischarge": PHYSICAL_TABLE_DIR / "FactDischarge.parquet",
    "DimHospital": PHYSICAL_TABLE_DIR / "DimHospital.parquet",
    "DimService": PHYSICAL_TABLE_DIR / "DimService.parquet",
    "DimCaseMix": PHYSICAL_TABLE_DIR / "DimCaseMix.parquet",
}

missing_files = [
    str(path)
    for path in [*input_files.values(), *physical_table_paths.values()]
    if not path.exists()
]

assert not missing_files, (
    "Required upstream outputs are missing:\n"
    + "\n".join(missing_files)
)

inputs = {name: pd.read_csv(path) for name, path in input_files.items()}

benchmark_specification = inputs["benchmark_specification"]
catalog_validation = inputs["catalog_validation"]
column_specification = inputs["column_specification"]
schema_validation = inputs["schema_validation"]
physical_validation = inputs["physical_validation"]
physical_parquet_validation = inputs["physical_parquet_validation"]

print("Benchmark specification rows:", len(benchmark_specification))
display(benchmark_specification)

Benchmark specification rows: 5


,benchmark_id,benchmark_name,outcome,primary_peer_keys,fallback_peer_keys,facility_excluded_from_own_peer,minimum_peer_n,peer_statistic,intended_use,not_intended_for,status
0,LOS_PEER_MEAN_PRIMARY,Primary Peer Mean LOS,LOS numeric lower bound,APR DRG Code|APR Severity of Illness Code,APR DRG Code,True,30.0,Arithmetic mean,Descriptive case-mix-contextualized LOS benchmark,Causal attribution or clinical LOS prediction,APPROVED_BASELINE
1,LOS_PEER_MEDIAN_CONTEXT,Peer Median LOS Context,LOS numeric lower bound,APR DRG Code|APR Severity of Illness Code,APR DRG Code,True,30.0,Median,Robust descriptive comparison alongside peer mean,Expected-total calculation or A/E denominator,DEFERRED_CONTEXT_IMPLEMENTATION
2,LOS_PREDICTIVE_MODEL,Model-Predicted Expected LOS,LOS modeling target,To be defined during modeling,Not applicable,True,NaN,Validated model prediction,Later case-mix-adjusted predictive benchmarking,Use before model validation and temporal evaluation,DEFERRED_TO_MODELING
3,COST_PEER_MEAN_PRIMARY,Primary Peer Mean Estimated Cost,Valid positive estimated cost,APR DRG Code|APR Severity of Illness Code,APR DRG Code,True,30.0,Arithmetic mean,Descriptive case-mix-contextualized estimated-cost benchmark,"Audited expense, reimbursement, profitability, or causal attribution",APPROVED_BASELINE
4,COST_PEER_MEDIAN_CONTEXT,Peer Median Estimated Cost Context,Valid positive estimated cost,APR DRG Code|APR Severity of Illness Code,APR DRG Code,True,30.0,Median,Robust descriptive estimated-cost comparison,Expected-total calculation or A/E denominator,DEFERRED_CONTEXT_IMPLEMENTATION


## 4. Upstream Commit Gates

In [4]:
def normalize_boolean(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )


assert normalize_boolean(catalog_validation["passed"]).fillna(False).all(), (
    "Notebook 02 validation results contain failures."
)
assert normalize_boolean(schema_validation["passed"]).fillna(False).all(), (
    "Notebook 03 schema validation results contain failures."
)
assert normalize_boolean(physical_validation["passed"]).fillna(False).all(), (
    "Notebook 04 physical validation results contain failures."
)
assert normalize_boolean(physical_parquet_validation["passed"]).fillna(False).all(), (
    "Notebook 04 Parquet validation results contain failures."
)
assert not benchmark_specification.empty, (
    "Notebook 02 benchmark specification is empty."
)

baseline_ids = [
    "LOS_PEER_MEAN_PRIMARY",
    "COST_PEER_MEAN_PRIMARY",
]

baseline_specs = (
    benchmark_specification
    .set_index("benchmark_id")
    .loc[baseline_ids]
)

assert baseline_specs["status"].eq("APPROVED_BASELINE").all()

assert baseline_specs["primary_peer_keys"].eq(
    "APR DRG Code|APR Severity of Illness Code"
).all()

assert baseline_specs["fallback_peer_keys"].eq(
    "APR DRG Code"
).all()

assert baseline_specs["peer_statistic"].eq(
    "Arithmetic mean"
).all()

assert normalize_boolean(
    baseline_specs["facility_excluded_from_own_peer"]
).fillna(False).all()

minimum_peer_values = (
    baseline_specs["minimum_peer_n"]
    .dropna()
    .astype(int)
    .unique()
)

assert len(minimum_peer_values) == 1, (
    "LOS and estimated-cost baseline benchmarks disagree on minimum_peer_n."
)

MIN_COMPARISON_N = int(minimum_peer_values[0])

print(
    "Notebook 02 benchmark contract validated. "
    f"Minimum comparison N = {MIN_COMPARISON_N}."
)

expected_benchmark_columns = {
    "peer_expected_los_days",
    "los_peer_comparison_n",
    "los_peer_benchmark_level",
    "peer_expected_estimated_cost",
    "cost_peer_comparison_n",
    "cost_peer_benchmark_level",
}

fact_spec_columns = set(
    column_specification.loc[
        column_specification["table_name"].eq("FactDischarge"),
        "column_name",
    ]
)

assert expected_benchmark_columns.issubset(fact_spec_columns), (
    "Notebook 03 no longer contains all required peer-benchmark fact columns."
)

print("Notebook 02, 03, and 04 commit gates passed.")

Notebook 02 benchmark contract validated. Minimum comparison N = 30.
Notebook 02, 03, and 04 commit gates passed.


### Interpretation

Notebook 05 does not redefine benchmark policy. Notebook 02 remains the benchmark-design authority, Notebook 03 remains the physical-schema authority, and Notebook 04 remains the validated pre-benchmark physical-model authority.

## 5. Benchmark Implementation Standards

In [5]:
PRIMARY_LEVEL = "APR-DRG × Severity"
FALLBACK_LEVEL = "APR-DRG"
UNAVAILABLE_LEVEL = "Unavailable"

benchmark_standards = pd.DataFrame([
    {
        "standard_id": "LEAVE_ONE_FACILITY_OUT",
        "decision": "All eligible records from the focal hospital are excluded from its peer expectation.",
        "rationale": "The facility being evaluated must not influence its own expected value.",
    },
    {
        "standard_id": "PRIMARY_PEER_GROUP",
        "decision": "Primary peer groups use APR-DRG Code and APR Severity of Illness Code.",
        "rationale": "This is the governed primary peer definition approved in Notebook 02.",
    },
    {
        "standard_id": "FALLBACK_PEER_GROUP",
        "decision": "APR-DRG Code alone is used when the primary comparison group is insufficient.",
        "rationale": "This preserves benchmark availability while retaining clinical grouping.",
    },
    {
        "standard_id": "MINIMUM_COMPARISON_N",
        "decision": f"At least {MIN_COMPARISON_N} eligible comparison discharges must remain after excluding the focal hospital.",
        "rationale": "Low-volume peer expectations are not treated as sufficiently stable for the governed benchmark.",
    },
    {
        "standard_id": "LOS_ELIGIBILITY",
        "decision": "LOS peer calculations use records where is_valid_los = 1.",
        "rationale": "The benchmark uses the governed LOS valid-record population.",
    },
    {
        "standard_id": "COST_ELIGIBILITY",
        "decision": "Estimated-cost peer calculations use records where is_valid_cost = 1.",
        "rationale": "The benchmark uses the governed estimated-cost valid-record population.",
    },
    {
        "standard_id": "FOCAL_RECORD_ELIGIBILITY",
        "decision": "A focal record receives a benchmark only when its corresponding measure is valid.",
        "rationale": "Expected values are not attached to records whose actual measure failed the governed validity rule.",
    },
    {
        "standard_id": "UNKNOWN_HOSPITAL",
        "decision": "Hospital key 0 does not receive a peer benchmark.",
        "rationale": "Leave-one-facility-out logic requires a resolved focal hospital.",
    },
    {
        "standard_id": "SERVICE_GRAIN_INDEPENDENCE",
        "decision": "DimService may use APR-DRG × APR-MDC as its physical natural key, but peer grouping uses APR-DRG as governed by Notebook 02.",
        "rationale": "Dimension grain and analytical benchmark grain serve different purposes.",
    },
    {
        "standard_id": "LOW_VOLUME_STORAGE",
        "decision": "When no benchmark qualifies, expectation remains null and benchmark level is labeled Unavailable.",
        "rationale": "Missing expected values must be distinguishable from valid zero values.",
    },
    {
        "standard_id": "DESCRIPTIVE_NOT_PREDICTIVE",
        "decision": "Peer expectations remain separate from future model-predicted LOS.",
        "rationale": "Descriptive benchmark values and machine-learning predictions have different interpretations.",
    },
])

display(benchmark_standards)

,standard_id,decision,rationale
0,LEAVE_ONE_FACILITY_OUT,All eligible records from the focal hospital are excluded from its peer expectation.,The facility being evaluated must not influence its own expected value.
1,PRIMARY_PEER_GROUP,Primary peer groups use APR-DRG Code and APR Severity of Illness Code.,This is the governed primary peer definition approved in Notebook 02.
2,FALLBACK_PEER_GROUP,APR-DRG Code alone is used when the primary comparison group is insufficient.,This preserves benchmark availability while retaining clinical grouping.
3,MINIMUM_COMPARISON_N,At least 30 eligible comparison discharges must remain after excluding the focal hospital.,Low-volume peer expectations are not treated as sufficiently stable for the governed benchmark.
4,LOS_ELIGIBILITY,LOS peer calculations use records where is_valid_los = 1.,The benchmark uses the governed LOS valid-record population.
5,COST_ELIGIBILITY,Estimated-cost peer calculations use records where is_valid_cost = 1.,The benchmark uses the governed estimated-cost valid-record population.
6,FOCAL_RECORD_ELIGIBILITY,A focal record receives a benchmark only when its corresponding measure is valid.,Expected values are not attached to records whose actual measure failed the governed validity rule.
7,UNKNOWN_HOSPITAL,Hospital key 0 does not receive a peer benchmark.,Leave-one-facility-out logic requires a resolved focal hospital.
8,SERVICE_GRAIN_INDEPENDENCE,"DimService may use APR-DRG × APR-MDC as its physical natural key, but peer grouping uses APR-DRG as governed by Notebook 02.",Dimension grain and analytical benchmark grain serve different purposes.
9,LOW_VOLUME_STORAGE,"When no benchmark qualifies, expectation remains null and benchmark level is labeled Unavailable.",Missing expected values must be distinguishable from valid zero values.


## 6. Load Physical Model with DuckDB

In [6]:
def sql_path_literal(path):
    return "'" + Path(path).resolve().as_posix().replace("'", "''") + "'"


def quote_identifier(value):
    return '"' + str(value).replace('"', '""') + '"'


if WORK_DB_PATH.exists():
    WORK_DB_PATH.unlink()

con = duckdb.connect(database=str(WORK_DB_PATH))

for table_name, table_path in physical_table_paths.items():
    view_name = f"{table_name}Input"
    con.execute(
        f"""
        CREATE OR REPLACE VIEW {quote_identifier(view_name)} AS
        SELECT *
        FROM read_parquet({sql_path_literal(table_path)})
        """
    )

fact_columns = (
    con.sql('DESCRIBE "FactDischargeInput"')
    .df()["column_name"]
    .tolist()
)

input_fact_row_count = int(
    con.sql(
        'SELECT COUNT(*) AS row_count FROM "FactDischargeInput"'
    ).df().loc[0, "row_count"]
)

print(f"Physical FactDischarge rows: {input_fact_row_count:,}")
print(f"FactDischarge columns: {len(fact_columns)}")

Physical FactDischarge rows: 2,125,754
FactDischarge columns: 24


## 7. Validate Benchmark Placeholders

In [7]:
benchmark_placeholder_profile = con.sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(
            CASE
                WHEN peer_expected_los_days IS NOT NULL
                  OR los_peer_comparison_n IS NOT NULL
                  OR los_peer_benchmark_level IS NOT NULL
                  OR peer_expected_estimated_cost IS NOT NULL
                  OR cost_peer_comparison_n IS NOT NULL
                  OR cost_peer_benchmark_level IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS prepopulated_benchmark_row_n
    FROM "FactDischargeInput"
    """
).df()

display(benchmark_placeholder_profile)

assert int(
    benchmark_placeholder_profile.loc[0, "prepopulated_benchmark_row_n"]
) == 0, (
    "Notebook 04 benchmark placeholder columns are already populated."
)

,total_rows,prepopulated_benchmark_row_n
0,2125754,0.0


## 8. Build Benchmark Context

The physical fact table stores surrogate keys rather than the APR-DRG and severity natural values needed by the benchmark definition. This section joins the validated physical dimensions back to the fact table for benchmark calculation only.

In [8]:
con.execute(
    """
    CREATE OR REPLACE VIEW benchmark_context AS
    SELECT
        f.*,
        h.permanent_facility_id,
        svc.apr_drg_code,
        cm.apr_severity_code
    FROM "FactDischargeInput" AS f
    LEFT JOIN "DimHospitalInput" AS h
        ON f.hospital_key = h.hospital_key
    LEFT JOIN "DimServiceInput" AS svc
        ON f.service_key = svc.service_key
    LEFT JOIN "DimCaseMixInput" AS cm
        ON f.case_mix_key = cm.case_mix_key
    """
)

benchmark_context_validation = con.sql(
    """
    SELECT
        COUNT(*) AS row_count,
        SUM(CASE WHEN hospital_key <> 0 AND permanent_facility_id IS NULL THEN 1 ELSE 0 END)
            AS unresolved_nonzero_hospital_n,
        SUM(CASE WHEN service_key <> 0 AND apr_drg_code IS NULL THEN 1 ELSE 0 END)
            AS unresolved_nonzero_service_n,
        SUM(CASE WHEN case_mix_key <> 0 AND apr_severity_code IS NULL THEN 1 ELSE 0 END)
            AS unresolved_nonzero_case_mix_n
    FROM benchmark_context
    """
).df()

display(benchmark_context_validation)
context_row = benchmark_context_validation.iloc[0]

assert int(context_row["row_count"]) == input_fact_row_count
assert int(context_row["unresolved_nonzero_hospital_n"]) == 0
assert int(context_row["unresolved_nonzero_service_n"]) == 0
assert int(context_row["unresolved_nonzero_case_mix_n"]) == 0

,row_count,unresolved_nonzero_hospital_n,unresolved_nonzero_service_n,unresolved_nonzero_case_mix_n
0,2125754,0.0,0.0,0.0


## 9. Benchmark Eligibility Profile

In [9]:
benchmark_eligibility = con.sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_valid_los = 1 THEN 1 ELSE 0 END) AS valid_los_n,
        SUM(CASE WHEN is_valid_los = 1 AND hospital_key <> 0 AND apr_drg_code IS NOT NULL THEN 1 ELSE 0 END)
            AS los_fallback_eligible_n,
        SUM(CASE WHEN is_valid_los = 1 AND hospital_key <> 0 AND apr_drg_code IS NOT NULL AND apr_severity_code IS NOT NULL THEN 1 ELSE 0 END)
            AS los_primary_eligible_n,
        SUM(CASE WHEN is_valid_cost = 1 THEN 1 ELSE 0 END) AS valid_cost_n,
        SUM(CASE WHEN is_valid_cost = 1 AND hospital_key <> 0 AND apr_drg_code IS NOT NULL THEN 1 ELSE 0 END)
            AS cost_fallback_eligible_n,
        SUM(CASE WHEN is_valid_cost = 1 AND hospital_key <> 0 AND apr_drg_code IS NOT NULL AND apr_severity_code IS NOT NULL THEN 1 ELSE 0 END)
            AS cost_primary_eligible_n
    FROM benchmark_context
    """
).df()

display(benchmark_eligibility)

,total_rows,valid_los_n,los_fallback_eligible_n,los_primary_eligible_n,valid_cost_n,cost_fallback_eligible_n,cost_primary_eligible_n
0,2125754,2125754.0,2120421.0,2119957.0,2125754.0,2120421.0,2119957.0


### Interpretation

Primary benchmark eligibility requires a resolved hospital, APR-DRG, severity, and valid measure. Fallback eligibility requires a resolved hospital, APR-DRG, and valid measure.

## 10. Calculate Primary LOS Peer Statistics

In [10]:
con.execute(
    """
    CREATE OR REPLACE TABLE los_primary_peer_stats AS
    WITH group_stats AS (
        SELECT
            apr_drg_code,
            apr_severity_code,
            COUNT(*) AS group_n,
            SUM(CAST(los_days_lower_bound AS DOUBLE)) AS group_sum
        FROM benchmark_context
        WHERE is_valid_los = 1
          AND hospital_key <> 0
          AND apr_drg_code IS NOT NULL
          AND apr_severity_code IS NOT NULL
        GROUP BY apr_drg_code, apr_severity_code
    ),
    hospital_stats AS (
        SELECT
            hospital_key,
            apr_drg_code,
            apr_severity_code,
            COUNT(*) AS hospital_n,
            SUM(CAST(los_days_lower_bound AS DOUBLE)) AS hospital_sum
        FROM benchmark_context
        WHERE is_valid_los = 1
          AND hospital_key <> 0
          AND apr_drg_code IS NOT NULL
          AND apr_severity_code IS NOT NULL
        GROUP BY hospital_key, apr_drg_code, apr_severity_code
    )
    SELECT
        h.hospital_key,
        h.apr_drg_code,
        h.apr_severity_code,
        CAST(g.group_n - h.hospital_n AS BIGINT) AS comparison_n,
        CASE
            WHEN (g.group_n - h.hospital_n) > 0
            THEN (g.group_sum - h.hospital_sum) / (g.group_n - h.hospital_n)
            ELSE NULL
        END AS peer_expected_los_days
    FROM hospital_stats AS h
    INNER JOIN group_stats AS g
        ON h.apr_drg_code = g.apr_drg_code
       AND h.apr_severity_code = g.apr_severity_code
    """
)

## 11. Calculate Fallback LOS Peer Statistics

In [11]:
con.execute(
    """
    CREATE OR REPLACE TABLE los_fallback_peer_stats AS
    WITH group_stats AS (
        SELECT
            apr_drg_code,
            COUNT(*) AS group_n,
            SUM(CAST(los_days_lower_bound AS DOUBLE)) AS group_sum
        FROM benchmark_context
        WHERE is_valid_los = 1
          AND hospital_key <> 0
          AND apr_drg_code IS NOT NULL
        GROUP BY apr_drg_code
    ),
    hospital_stats AS (
        SELECT
            hospital_key,
            apr_drg_code,
            COUNT(*) AS hospital_n,
            SUM(CAST(los_days_lower_bound AS DOUBLE)) AS hospital_sum
        FROM benchmark_context
        WHERE is_valid_los = 1
          AND hospital_key <> 0
          AND apr_drg_code IS NOT NULL
        GROUP BY hospital_key, apr_drg_code
    )
    SELECT
        h.hospital_key,
        h.apr_drg_code,
        CAST(g.group_n - h.hospital_n AS BIGINT) AS comparison_n,
        CASE
            WHEN (g.group_n - h.hospital_n) > 0
            THEN (g.group_sum - h.hospital_sum) / (g.group_n - h.hospital_n)
            ELSE NULL
        END AS peer_expected_los_days
    FROM hospital_stats AS h
    INNER JOIN group_stats AS g
        ON h.apr_drg_code = g.apr_drg_code
    """
)

## 12. Calculate Primary Estimated-Cost Peer Statistics

In [12]:
con.execute(
    """
    CREATE OR REPLACE TABLE cost_primary_peer_stats AS
    WITH group_stats AS (
        SELECT
            apr_drg_code,
            apr_severity_code,
            COUNT(*) AS group_n,
            SUM(CAST(total_costs AS DECIMAL(38, 2))) AS group_sum
        FROM benchmark_context
        WHERE is_valid_cost = 1
          AND hospital_key <> 0
          AND apr_drg_code IS NOT NULL
          AND apr_severity_code IS NOT NULL
        GROUP BY apr_drg_code, apr_severity_code
    ),
    hospital_stats AS (
        SELECT
            hospital_key,
            apr_drg_code,
            apr_severity_code,
            COUNT(*) AS hospital_n,
            SUM(CAST(total_costs AS DECIMAL(38, 2))) AS hospital_sum
        FROM benchmark_context
        WHERE is_valid_cost = 1
          AND hospital_key <> 0
          AND apr_drg_code IS NOT NULL
          AND apr_severity_code IS NOT NULL
        GROUP BY hospital_key, apr_drg_code, apr_severity_code
    )
    SELECT
        h.hospital_key,
        h.apr_drg_code,
        h.apr_severity_code,
        CAST(g.group_n - h.hospital_n AS BIGINT) AS comparison_n,
        CASE
            WHEN (g.group_n - h.hospital_n) > 0
            THEN CAST(
                (g.group_sum - h.hospital_sum) / (g.group_n - h.hospital_n)
                AS DECIMAL(18, 2)
            )
            ELSE NULL
        END AS peer_expected_estimated_cost
    FROM hospital_stats AS h
    INNER JOIN group_stats AS g
        ON h.apr_drg_code = g.apr_drg_code
       AND h.apr_severity_code = g.apr_severity_code
    """
)

## 13. Calculate Fallback Estimated-Cost Peer Statistics

In [13]:
con.execute(
    """
    CREATE OR REPLACE TABLE cost_fallback_peer_stats AS
    WITH group_stats AS (
        SELECT
            apr_drg_code,
            COUNT(*) AS group_n,
            SUM(CAST(total_costs AS DECIMAL(38, 2))) AS group_sum
        FROM benchmark_context
        WHERE is_valid_cost = 1
          AND hospital_key <> 0
          AND apr_drg_code IS NOT NULL
        GROUP BY apr_drg_code
    ),
    hospital_stats AS (
        SELECT
            hospital_key,
            apr_drg_code,
            COUNT(*) AS hospital_n,
            SUM(CAST(total_costs AS DECIMAL(38, 2))) AS hospital_sum
        FROM benchmark_context
        WHERE is_valid_cost = 1
          AND hospital_key <> 0
          AND apr_drg_code IS NOT NULL
        GROUP BY hospital_key, apr_drg_code
    )
    SELECT
        h.hospital_key,
        h.apr_drg_code,
        CAST(g.group_n - h.hospital_n AS BIGINT) AS comparison_n,
        CASE
            WHEN (g.group_n - h.hospital_n) > 0
            THEN CAST(
                (g.group_sum - h.hospital_sum) / (g.group_n - h.hospital_n)
                AS DECIMAL(18, 2)
            )
            ELSE NULL
        END AS peer_expected_estimated_cost
    FROM hospital_stats AS h
    INNER JOIN group_stats AS g
        ON h.apr_drg_code = g.apr_drg_code
    """
)

## 14. Resolve Primary vs Fallback Benchmark

In [14]:
con.execute(
    f"""
    CREATE OR REPLACE TABLE benchmark_assignment AS
    SELECT
        c.*,
        lp.comparison_n AS los_primary_comparison_n,
        lp.peer_expected_los_days AS los_primary_expected,
        lf.comparison_n AS los_fallback_comparison_n,
        lf.peer_expected_los_days AS los_fallback_expected,
        cp.comparison_n AS cost_primary_comparison_n,
        cp.peer_expected_estimated_cost AS cost_primary_expected,
        cf.comparison_n AS cost_fallback_comparison_n,
        cf.peer_expected_estimated_cost AS cost_fallback_expected,

        CASE
            WHEN c.is_valid_los <> 1 THEN NULL
            WHEN c.hospital_key = 0 THEN NULL
            WHEN c.apr_drg_code IS NULL THEN NULL
            WHEN lp.comparison_n >= {MIN_COMPARISON_N} THEN lp.peer_expected_los_days
            WHEN lf.comparison_n >= {MIN_COMPARISON_N} THEN lf.peer_expected_los_days
            ELSE NULL
        END AS resolved_peer_expected_los_days,

        CASE
            WHEN c.is_valid_los <> 1 THEN NULL
            WHEN c.hospital_key = 0 THEN NULL
            WHEN c.apr_drg_code IS NULL THEN NULL
            WHEN lp.comparison_n >= {MIN_COMPARISON_N} THEN lp.comparison_n
            WHEN lf.comparison_n >= {MIN_COMPARISON_N} THEN lf.comparison_n
            ELSE lf.comparison_n
        END AS resolved_los_peer_comparison_n,

        CASE
            WHEN c.is_valid_los <> 1 THEN '{UNAVAILABLE_LEVEL}'
            WHEN c.hospital_key = 0 THEN '{UNAVAILABLE_LEVEL}'
            WHEN c.apr_drg_code IS NULL THEN '{UNAVAILABLE_LEVEL}'
            WHEN lp.comparison_n >= {MIN_COMPARISON_N} THEN '{PRIMARY_LEVEL}'
            WHEN lf.comparison_n >= {MIN_COMPARISON_N} THEN '{FALLBACK_LEVEL}'
            ELSE '{UNAVAILABLE_LEVEL}'
        END AS resolved_los_benchmark_level,

        CASE
            WHEN c.is_valid_los <> 1 THEN 'Invalid focal LOS'
            WHEN c.hospital_key = 0 THEN 'Unresolved hospital'
            WHEN c.apr_drg_code IS NULL THEN 'Missing APR-DRG'
            WHEN lp.comparison_n >= {MIN_COMPARISON_N} THEN 'Primary benchmark available'
            WHEN lf.comparison_n >= {MIN_COMPARISON_N} THEN 'Fallback benchmark available'
            ELSE 'Insufficient comparison volume'
        END AS los_benchmark_status,

        CASE
            WHEN c.is_valid_cost <> 1 THEN NULL
            WHEN c.hospital_key = 0 THEN NULL
            WHEN c.apr_drg_code IS NULL THEN NULL
            WHEN cp.comparison_n >= {MIN_COMPARISON_N} THEN cp.peer_expected_estimated_cost
            WHEN cf.comparison_n >= {MIN_COMPARISON_N} THEN cf.peer_expected_estimated_cost
            ELSE NULL
        END AS resolved_peer_expected_estimated_cost,

        CASE
            WHEN c.is_valid_cost <> 1 THEN NULL
            WHEN c.hospital_key = 0 THEN NULL
            WHEN c.apr_drg_code IS NULL THEN NULL
            WHEN cp.comparison_n >= {MIN_COMPARISON_N} THEN cp.comparison_n
            WHEN cf.comparison_n >= {MIN_COMPARISON_N} THEN cf.comparison_n
            ELSE cf.comparison_n
        END AS resolved_cost_peer_comparison_n,

        CASE
            WHEN c.is_valid_cost <> 1 THEN '{UNAVAILABLE_LEVEL}'
            WHEN c.hospital_key = 0 THEN '{UNAVAILABLE_LEVEL}'
            WHEN c.apr_drg_code IS NULL THEN '{UNAVAILABLE_LEVEL}'
            WHEN cp.comparison_n >= {MIN_COMPARISON_N} THEN '{PRIMARY_LEVEL}'
            WHEN cf.comparison_n >= {MIN_COMPARISON_N} THEN '{FALLBACK_LEVEL}'
            ELSE '{UNAVAILABLE_LEVEL}'
        END AS resolved_cost_benchmark_level,

        CASE
            WHEN c.is_valid_cost <> 1 THEN 'Invalid focal estimated cost'
            WHEN c.hospital_key = 0 THEN 'Unresolved hospital'
            WHEN c.apr_drg_code IS NULL THEN 'Missing APR-DRG'
            WHEN cp.comparison_n >= {MIN_COMPARISON_N} THEN 'Primary benchmark available'
            WHEN cf.comparison_n >= {MIN_COMPARISON_N} THEN 'Fallback benchmark available'
            ELSE 'Insufficient comparison volume'
        END AS cost_benchmark_status

    FROM benchmark_context AS c
    LEFT JOIN los_primary_peer_stats AS lp
        ON c.hospital_key = lp.hospital_key
       AND c.apr_drg_code = lp.apr_drg_code
       AND c.apr_severity_code = lp.apr_severity_code
    LEFT JOIN los_fallback_peer_stats AS lf
        ON c.hospital_key = lf.hospital_key
       AND c.apr_drg_code = lf.apr_drg_code
    LEFT JOIN cost_primary_peer_stats AS cp
        ON c.hospital_key = cp.hospital_key
       AND c.apr_drg_code = cp.apr_drg_code
       AND c.apr_severity_code = cp.apr_severity_code
    LEFT JOIN cost_fallback_peer_stats AS cf
        ON c.hospital_key = cf.hospital_key
       AND c.apr_drg_code = cf.apr_drg_code
    """
)

## 15. Build Benchmarked FactDischarge

In [15]:
resolved_column_expressions = {
    "peer_expected_los_days": (
        "CAST(resolved_peer_expected_los_days AS DOUBLE) AS peer_expected_los_days"
    ),
    "los_peer_comparison_n": (
        "CAST(resolved_los_peer_comparison_n AS BIGINT) AS los_peer_comparison_n"
    ),
    "los_peer_benchmark_level": (
        "CAST(resolved_los_benchmark_level AS VARCHAR) AS los_peer_benchmark_level"
    ),
    "peer_expected_estimated_cost": (
        "CAST(resolved_peer_expected_estimated_cost AS DECIMAL(18, 2)) AS peer_expected_estimated_cost"
    ),
    "cost_peer_comparison_n": (
        "CAST(resolved_cost_peer_comparison_n AS BIGINT) AS cost_peer_comparison_n"
    ),
    "cost_peer_benchmark_level": (
        "CAST(resolved_cost_benchmark_level AS VARCHAR) AS cost_peer_benchmark_level"
    ),
}

fact_select_expressions = []
for column_name in fact_columns:
    if column_name in resolved_column_expressions:
        fact_select_expressions.append(resolved_column_expressions[column_name])
    else:
        fact_select_expressions.append(quote_identifier(column_name))

fact_select_sql = ",\n        ".join(fact_select_expressions)

con.execute(
    f"""
    CREATE OR REPLACE TABLE "FactDischargeBenchmarked" AS
    SELECT
        {fact_select_sql}
    FROM benchmark_assignment
    """
)

output_fact_row_count = int(
    con.sql(
        'SELECT COUNT(*) AS row_count FROM "FactDischargeBenchmarked"'
    ).df().loc[0, "row_count"]
)

assert output_fact_row_count == input_fact_row_count, (
    "Benchmark calculation changed the FactDischarge row count."
)

print(f"Benchmarked FactDischarge rows: {output_fact_row_count:,}")

Benchmarked FactDischarge rows: 2,125,754


## 16. Benchmark Coverage Diagnostics

In [16]:
benchmark_coverage = con.sql(
    """
    SELECT
        'LOS' AS benchmark_metric,
        resolved_los_benchmark_level AS benchmark_level,
        los_benchmark_status AS benchmark_status,
        COUNT(*) AS row_n,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS row_pct
    FROM benchmark_assignment
    GROUP BY resolved_los_benchmark_level, los_benchmark_status

    UNION ALL

    SELECT
        'Estimated Cost' AS benchmark_metric,
        resolved_cost_benchmark_level AS benchmark_level,
        cost_benchmark_status AS benchmark_status,
        COUNT(*) AS row_n,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS row_pct
    FROM benchmark_assignment
    GROUP BY resolved_cost_benchmark_level, cost_benchmark_status

    ORDER BY benchmark_metric, benchmark_level, benchmark_status
    """
).df()

display(benchmark_coverage)

benchmark_peer_size_summary = con.sql(
    """
    SELECT
        'LOS' AS benchmark_metric,
        resolved_los_benchmark_level AS benchmark_level,
        COUNT(*) AS benchmarked_row_n,
        MIN(resolved_los_peer_comparison_n) AS min_peer_n,
        MEDIAN(resolved_los_peer_comparison_n) AS median_peer_n,
        QUANTILE_CONT(resolved_los_peer_comparison_n, 0.95) AS p95_peer_n,
        MAX(resolved_los_peer_comparison_n) AS max_peer_n
    FROM benchmark_assignment
    WHERE resolved_peer_expected_los_days IS NOT NULL
    GROUP BY resolved_los_benchmark_level

    UNION ALL

    SELECT
        'Estimated Cost' AS benchmark_metric,
        resolved_cost_benchmark_level AS benchmark_level,
        COUNT(*) AS benchmarked_row_n,
        MIN(resolved_cost_peer_comparison_n) AS min_peer_n,
        MEDIAN(resolved_cost_peer_comparison_n) AS median_peer_n,
        QUANTILE_CONT(resolved_cost_peer_comparison_n, 0.95) AS p95_peer_n,
        MAX(resolved_cost_peer_comparison_n) AS max_peer_n
    FROM benchmark_assignment
    WHERE resolved_peer_expected_estimated_cost IS NOT NULL
    GROUP BY resolved_cost_benchmark_level

    ORDER BY benchmark_metric, benchmark_level
    """
).df()

display(benchmark_peer_size_summary)

,benchmark_metric,benchmark_level,benchmark_status,row_n,row_pct
0,Estimated Cost,APR-DRG,Fallback benchmark available,2570,0.1209
1,Estimated Cost,APR-DRG × Severity,Primary benchmark available,2117792,99.6255
2,Estimated Cost,Unavailable,Insufficient comparison volume,59,0.0028
3,Estimated Cost,Unavailable,Unresolved hospital,5333,0.2509
4,LOS,APR-DRG,Fallback benchmark available,2570,0.1209
5,LOS,APR-DRG × Severity,Primary benchmark available,2117792,99.6255
6,LOS,Unavailable,Insufficient comparison volume,59,0.0028
7,LOS,Unavailable,Unresolved hospital,5333,0.2509


,benchmark_metric,benchmark_level,benchmarked_row_n,min_peer_n,median_peer_n,p95_peer_n,max_peer_n
0,Estimated Cost,APR-DRG,2570,30,375.0,4731.4,164903
1,Estimated Cost,APR-DRG × Severity,2117792,30,5162.0,118935.0,122766
2,LOS,APR-DRG,2570,30,375.0,4731.4,164903
3,LOS,APR-DRG × Severity,2117792,30,5162.0,118935.0,122766


## 17. Validate Benchmark Selection Logic

In [17]:
selection_logic_validation = con.sql(
    f"""
    SELECT
        SUM(CASE WHEN resolved_peer_expected_los_days IS NOT NULL
                  AND resolved_los_peer_comparison_n < {MIN_COMPARISON_N}
                 THEN 1 ELSE 0 END)
            AS los_benchmark_below_minimum_n,

        SUM(CASE WHEN is_valid_los = 1
                  AND hospital_key <> 0
                  AND apr_drg_code IS NOT NULL
                  AND apr_severity_code IS NOT NULL
                  AND los_primary_comparison_n >= {MIN_COMPARISON_N}
                  AND resolved_los_benchmark_level <> '{PRIMARY_LEVEL}'
                 THEN 1 ELSE 0 END)
            AS los_primary_not_selected_when_available_n,

        SUM(CASE WHEN resolved_los_benchmark_level = '{FALLBACK_LEVEL}'
                  AND (
                      COALESCE(los_primary_comparison_n, -1) >= {MIN_COMPARISON_N}
                      OR COALESCE(los_fallback_comparison_n, -1) < {MIN_COMPARISON_N}
                  )
                 THEN 1 ELSE 0 END)
            AS los_invalid_fallback_selection_n,

        SUM(CASE WHEN is_valid_los <> 1
                  AND resolved_peer_expected_los_days IS NOT NULL
                 THEN 1 ELSE 0 END)
            AS invalid_los_received_benchmark_n,

        SUM(CASE WHEN resolved_peer_expected_los_days <= 0 THEN 1 ELSE 0 END)
            AS nonpositive_los_expectation_n,

        SUM(CASE WHEN resolved_peer_expected_estimated_cost IS NOT NULL
                  AND resolved_cost_peer_comparison_n < {MIN_COMPARISON_N}
                 THEN 1 ELSE 0 END)
            AS cost_benchmark_below_minimum_n,

        SUM(CASE WHEN is_valid_cost = 1
                  AND hospital_key <> 0
                  AND apr_drg_code IS NOT NULL
                  AND apr_severity_code IS NOT NULL
                  AND cost_primary_comparison_n >= {MIN_COMPARISON_N}
                  AND resolved_cost_benchmark_level <> '{PRIMARY_LEVEL}'
                 THEN 1 ELSE 0 END)
            AS cost_primary_not_selected_when_available_n,

        SUM(CASE WHEN resolved_cost_benchmark_level = '{FALLBACK_LEVEL}'
                  AND (
                      COALESCE(cost_primary_comparison_n, -1) >= {MIN_COMPARISON_N}
                      OR COALESCE(cost_fallback_comparison_n, -1) < {MIN_COMPARISON_N}
                  )
                 THEN 1 ELSE 0 END)
            AS cost_invalid_fallback_selection_n,

        SUM(CASE WHEN is_valid_cost <> 1
                  AND resolved_peer_expected_estimated_cost IS NOT NULL
                 THEN 1 ELSE 0 END)
            AS invalid_cost_received_benchmark_n,

        SUM(CASE WHEN resolved_peer_expected_estimated_cost <= 0 THEN 1 ELSE 0 END)
            AS nonpositive_cost_expectation_n

    FROM benchmark_assignment
    """
).df()

display(selection_logic_validation)

,los_benchmark_below_minimum_n,los_primary_not_selected_when_available_n,los_invalid_fallback_selection_n,invalid_los_received_benchmark_n,nonpositive_los_expectation_n,cost_benchmark_below_minimum_n,cost_primary_not_selected_when_available_n,cost_invalid_fallback_selection_n,invalid_cost_received_benchmark_n,nonpositive_cost_expectation_n
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 18. Validate Allowed Benchmark-Level Values

In [18]:
allowed_levels = {PRIMARY_LEVEL, FALLBACK_LEVEL, UNAVAILABLE_LEVEL}

los_levels = set(
    con.sql(
        'SELECT DISTINCT los_peer_benchmark_level FROM "FactDischargeBenchmarked"'
    ).df()["los_peer_benchmark_level"].dropna()
)

cost_levels = set(
    con.sql(
        'SELECT DISTINCT cost_peer_benchmark_level FROM "FactDischargeBenchmarked"'
    ).df()["cost_peer_benchmark_level"].dropna()
)

assert los_levels.issubset(allowed_levels)
assert cost_levels.issubset(allowed_levels)

print("LOS benchmark levels:", sorted(los_levels))
print("Cost benchmark levels:", sorted(cost_levels))

LOS benchmark levels: ['APR-DRG', 'APR-DRG × Severity', 'Unavailable']
Cost benchmark levels: ['APR-DRG', 'APR-DRG × Severity', 'Unavailable']


## 19. Validate Non-Benchmark Fact Columns Remain Unchanged

Because the source does not contain a durable discharge identifier, row-level identity is not invented for reconciliation. Instead, this validation compares the complete multiset of all non-benchmark fact columns before and after benchmark assignment.

In [19]:
nonbenchmark_columns = [
    column
    for column in fact_columns
    if column not in expected_benchmark_columns
]

nonbenchmark_column_sql = ", ".join(
    quote_identifier(column)
    for column in nonbenchmark_columns
)

input_minus_output_n = int(
    con.sql(
        f"""
        SELECT COUNT(*) AS mismatch_n
        FROM (
            SELECT {nonbenchmark_column_sql}
            FROM "FactDischargeInput"
            EXCEPT ALL
            SELECT {nonbenchmark_column_sql}
            FROM "FactDischargeBenchmarked"
        )
        """
    ).df().loc[0, "mismatch_n"]
)

output_minus_input_n = int(
    con.sql(
        f"""
        SELECT COUNT(*) AS mismatch_n
        FROM (
            SELECT {nonbenchmark_column_sql}
            FROM "FactDischargeBenchmarked"
            EXCEPT ALL
            SELECT {nonbenchmark_column_sql}
            FROM "FactDischargeInput"
        )
        """
    ).df().loc[0, "mismatch_n"]
)

nonbenchmark_integrity = pd.DataFrame([
    {
        "validation_test": "Input non-benchmark rows missing from output",
        "mismatch_n": input_minus_output_n,
        "passed": input_minus_output_n == 0,
    },
    {
        "validation_test": "Output non-benchmark rows absent from input",
        "mismatch_n": output_minus_input_n,
        "passed": output_minus_input_n == 0,
    },
])

display(nonbenchmark_integrity)
assert nonbenchmark_integrity["passed"].all()

,validation_test,mismatch_n,passed
0,Input non-benchmark rows missing from output,0,True
1,Output non-benchmark rows absent from input,0,True


## 20. Direct Mathematical Reconciliation

A deterministic sample of benchmarked hospital-peer combinations is recalculated using direct AVG() queries that explicitly exclude the focal hospital. This provides a second implementation path for validating the aggregate subtraction method.

In [20]:
los_sample_contexts = con.sql(
    """
    WITH sample_source AS (
        SELECT DISTINCT
            hospital_key,
            apr_drg_code,
            apr_severity_code,
            resolved_los_benchmark_level AS benchmark_level,
            resolved_los_peer_comparison_n AS saved_peer_n,
            resolved_peer_expected_los_days AS saved_expected
        FROM benchmark_assignment
        WHERE resolved_peer_expected_los_days IS NOT NULL
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY benchmark_level
                ORDER BY hospital_key, apr_drg_code, apr_severity_code
            ) AS rn
        FROM sample_source
    )
    SELECT
        hospital_key,
        apr_drg_code,
        apr_severity_code,
        benchmark_level,
        saved_peer_n,
        saved_expected
    FROM ranked
    WHERE rn <= 3
    ORDER BY benchmark_level, hospital_key, apr_drg_code, apr_severity_code
    """
).df()

cost_sample_contexts = con.sql(
    """
    WITH sample_source AS (
        SELECT DISTINCT
            hospital_key,
            apr_drg_code,
            apr_severity_code,
            resolved_cost_benchmark_level AS benchmark_level,
            resolved_cost_peer_comparison_n AS saved_peer_n,
            resolved_peer_expected_estimated_cost AS saved_expected
        FROM benchmark_assignment
        WHERE resolved_peer_expected_estimated_cost IS NOT NULL
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY benchmark_level
                ORDER BY hospital_key, apr_drg_code, apr_severity_code
            ) AS rn
        FROM sample_source
    )
    SELECT
        hospital_key,
        apr_drg_code,
        apr_severity_code,
        benchmark_level,
        saved_peer_n,
        saved_expected
    FROM ranked
    WHERE rn <= 3
    ORDER BY benchmark_level, hospital_key, apr_drg_code, apr_severity_code
    """
).df()

In [21]:
manual_reconciliation_rows = []

for _, row in los_sample_contexts.iterrows():
    hospital_key = int(row["hospital_key"])
    apr_drg_code = row["apr_drg_code"]
    benchmark_level = row["benchmark_level"]

    if benchmark_level == PRIMARY_LEVEL:
        result = con.execute(
            """
            SELECT
                COUNT(*) AS peer_n,
                AVG(CAST(los_days_lower_bound AS DOUBLE)) AS direct_expected
            FROM benchmark_context
            WHERE is_valid_los = 1
              AND hospital_key <> 0
              AND hospital_key <> ?
              AND apr_drg_code = ?
              AND apr_severity_code = ?
            """,
            [hospital_key, apr_drg_code, int(row["apr_severity_code"])],
        ).df().iloc[0]
    else:
        result = con.execute(
            """
            SELECT
                COUNT(*) AS peer_n,
                AVG(CAST(los_days_lower_bound AS DOUBLE)) AS direct_expected
            FROM benchmark_context
            WHERE is_valid_los = 1
              AND hospital_key <> 0
              AND hospital_key <> ?
              AND apr_drg_code = ?
            """,
            [hospital_key, apr_drg_code],
        ).df().iloc[0]

    saved_expected = float(row["saved_expected"])
    direct_expected = float(result["direct_expected"])

    manual_reconciliation_rows.append({
        "benchmark_metric": "LOS",
        "hospital_key": hospital_key,
        "apr_drg_code": apr_drg_code,
        "apr_severity_code": row["apr_severity_code"],
        "benchmark_level": benchmark_level,
        "saved_peer_n": int(row["saved_peer_n"]),
        "direct_peer_n": int(result["peer_n"]),
        "saved_expected": saved_expected,
        "direct_expected": direct_expected,
        "absolute_difference": abs(saved_expected - direct_expected),
        "passed": (
            int(row["saved_peer_n"]) == int(result["peer_n"])
            and abs(saved_expected - direct_expected) < 1e-9
        ),
    })

for _, row in cost_sample_contexts.iterrows():
    hospital_key = int(row["hospital_key"])
    apr_drg_code = row["apr_drg_code"]
    benchmark_level = row["benchmark_level"]

    if benchmark_level == PRIMARY_LEVEL:
        result = con.execute(
            """
            SELECT
                COUNT(*) AS peer_n,
                ROUND(AVG(CAST(total_costs AS DOUBLE)), 2) AS direct_expected
            FROM benchmark_context
            WHERE is_valid_cost = 1
              AND hospital_key <> 0
              AND hospital_key <> ?
              AND apr_drg_code = ?
              AND apr_severity_code = ?
            """,
            [hospital_key, apr_drg_code, int(row["apr_severity_code"])],
        ).df().iloc[0]
    else:
        result = con.execute(
            """
            SELECT
                COUNT(*) AS peer_n,
                ROUND(AVG(CAST(total_costs AS DOUBLE)), 2) AS direct_expected
            FROM benchmark_context
            WHERE is_valid_cost = 1
              AND hospital_key <> 0
              AND hospital_key <> ?
              AND apr_drg_code = ?
            """,
            [hospital_key, apr_drg_code],
        ).df().iloc[0]

    saved_expected = float(row["saved_expected"])
    direct_expected = float(result["direct_expected"])

    manual_reconciliation_rows.append({
        "benchmark_metric": "Estimated Cost",
        "hospital_key": hospital_key,
        "apr_drg_code": apr_drg_code,
        "apr_severity_code": row["apr_severity_code"],
        "benchmark_level": benchmark_level,
        "saved_peer_n": int(row["saved_peer_n"]),
        "direct_peer_n": int(result["peer_n"]),
        "saved_expected": saved_expected,
        "direct_expected": direct_expected,
        "absolute_difference": abs(saved_expected - direct_expected),
        "passed": (
            int(row["saved_peer_n"]) == int(result["peer_n"])
            and abs(saved_expected - direct_expected) <= 0.01
        ),
    })

manual_reconciliation = pd.DataFrame(manual_reconciliation_rows)
display(manual_reconciliation)

assert not manual_reconciliation.empty
assert manual_reconciliation["passed"].all(), (
    "One or more direct peer benchmark reconciliation tests failed."
)

,benchmark_metric,hospital_key,apr_drg_code,apr_severity_code,benchmark_level,saved_peer_n,direct_peer_n,saved_expected,direct_expected,absolute_difference,passed
0,LOS,1,005,2,APR-DRG,2611,2611,47.883953,47.883953,0.0,True
1,LOS,1,046,4,APR-DRG,1507,1507,3.695421,3.695421,0.0,True
2,LOS,1,056,1,APR-DRG,92,92,4.663043,4.663043,0.0,True
3,LOS,1,004,3,APR-DRG × Severity,220,220,37.000000,37.000000,0.0,True
4,LOS,1,004,4,APR-DRG × Severity,2018,2018,58.128345,58.128345,0.0,True
5,LOS,1,005,3,APR-DRG × Severity,275,275,36.120000,36.120000,0.0,True
6,Estimated Cost,1,005,2,APR-DRG,2611,2611,297519.500000,297519.500000,0.0,True
7,Estimated Cost,1,046,4,APR-DRG,1507,1507,15984.350000,15984.350000,0.0,True
8,Estimated Cost,1,056,1,APR-DRG,92,92,21314.570000,21314.570000,0.0,True
9,Estimated Cost,1,004,3,APR-DRG × Severity,220,220,250392.520000,250392.520000,0.0,True


## 21. Physical Schema Validation

In [22]:
output_fact_columns = (
    con.sql('DESCRIBE "FactDischargeBenchmarked"')
    .df()["column_name"]
    .tolist()
)

schema_matches = fact_columns == output_fact_columns

fact_schema_validation = pd.DataFrame([
    {
        "validation_test": "Benchmarked fact schema matches Notebook 04 fact schema",
        "expected_column_n": len(fact_columns),
        "actual_column_n": len(output_fact_columns),
        "passed": schema_matches,
    }
])

display(fact_schema_validation)
assert schema_matches

,validation_test,expected_column_n,actual_column_n,passed
0,Benchmarked fact schema matches Notebook 04 fact schema,24,24,True


## 22. Consolidated Benchmark Validation Gates

In [23]:
selection_error_n = int(selection_logic_validation.iloc[0].sum())

available_los_n = int(
    con.sql(
        'SELECT COUNT(*) AS row_n FROM "FactDischargeBenchmarked" WHERE peer_expected_los_days IS NOT NULL'
    ).df().loc[0, "row_n"]
)

available_cost_n = int(
    con.sql(
        'SELECT COUNT(*) AS row_n FROM "FactDischargeBenchmarked" WHERE peer_expected_estimated_cost IS NOT NULL'
    ).df().loc[0, "row_n"]
)

benchmark_validation_results = pd.DataFrame([
    {
        "validation_test": "Notebook 02 validation passed",
        "passed": normalize_boolean(catalog_validation["passed"]).fillna(False).all(),
        "details": "",
    },
    {
        "validation_test": "Notebook 03 validation passed",
        "passed": normalize_boolean(schema_validation["passed"]).fillna(False).all(),
        "details": "",
    },
    {
        "validation_test": "Notebook 04 physical validation passed",
        "passed": normalize_boolean(physical_validation["passed"]).fillna(False).all(),
        "details": "",
    },
    {
        "validation_test": "Input benchmark columns were empty",
        "passed": int(benchmark_placeholder_profile.loc[0, "prepopulated_benchmark_row_n"]) == 0,
        "details": "",
    },
    {
        "validation_test": "Benchmark context preserves fact row count",
        "passed": int(context_row["row_count"]) == input_fact_row_count,
        "details": f"{int(context_row['row_count'])} vs {input_fact_row_count}",
    },
    {
        "validation_test": "Benchmarked fact preserves fact row count",
        "passed": output_fact_row_count == input_fact_row_count,
        "details": f"{output_fact_row_count} vs {input_fact_row_count}",
    },
    {
        "validation_test": "Non-benchmark fact columns are unchanged",
        "passed": nonbenchmark_integrity["passed"].all(),
        "details": f"{input_minus_output_n} input-only; {output_minus_input_n} output-only",
    },
    {
        "validation_test": "Benchmark selection logic has no violations",
        "passed": selection_error_n == 0,
        "details": str(selection_error_n),
    },
    {
        "validation_test": "Direct mathematical reconciliation passes",
        "passed": manual_reconciliation["passed"].all(),
        "details": f"{len(manual_reconciliation)} sample comparisons",
    },
    {
        "validation_test": "Fact schema remains unchanged",
        "passed": schema_matches,
        "details": "",
    },
    {
        "validation_test": "At least one LOS benchmark is available",
        "passed": available_los_n > 0,
        "details": str(available_los_n),
    },
    {
        "validation_test": "At least one estimated-cost benchmark is available",
        "passed": available_cost_n > 0,
        "details": str(available_cost_n),
    },
])

display(benchmark_validation_results)

assert benchmark_validation_results["passed"].all(), (
    "Notebook 05 failed one or more benchmark validation gates."
)

print("All peer-benchmark validation gates passed.")

,validation_test,passed,details
0,Notebook 02 validation passed,True,
1,Notebook 03 validation passed,True,
2,Notebook 04 physical validation passed,True,
3,Input benchmark columns were empty,True,
4,Benchmark context preserves fact row count,True,2125754 vs 2125754
5,Benchmarked fact preserves fact row count,True,2125754 vs 2125754
6,Non-benchmark fact columns are unchanged,True,0 input-only; 0 output-only
7,Benchmark selection logic has no violations,True,0
8,Direct mathematical reconciliation passes,True,12 sample comparisons
9,Fact schema remains unchanged,True,


All peer-benchmark validation gates passed.


## 23. Export Benchmarked FactDischarge

Notebook 04's physical-model fact table is not overwritten. Notebook 05 creates a downstream benchmarked fact table so the pre-benchmark physical layer remains reproducible and independently auditable.

In [24]:
BENCHMARKED_FACT_PATH = TABLE_DIR / "FactDischarge.parquet"

if BENCHMARKED_FACT_PATH.exists():
    BENCHMARKED_FACT_PATH.unlink()

con.execute(
    f"""
    COPY "FactDischargeBenchmarked"
    TO {sql_path_literal(BENCHMARKED_FACT_PATH)}
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

assert BENCHMARKED_FACT_PATH.exists()
assert BENCHMARKED_FACT_PATH.stat().st_size > 0

print(
    "Benchmarked fact table created:",
    BENCHMARKED_FACT_PATH.relative_to(PROJECT_ROOT),
)

Benchmarked fact table created: outputs\peer_benchmarks\tables\FactDischarge.parquet


## 24. Validate Exported Parquet

In [25]:
exported_fact_row_count = int(
    con.sql(
        f"""
        SELECT COUNT(*) AS row_count
        FROM read_parquet({sql_path_literal(BENCHMARKED_FACT_PATH)})
        """
    ).df().loc[0, "row_count"]
)

exported_fact_columns = (
    con.sql(
        f"""
        DESCRIBE
        SELECT *
        FROM read_parquet({sql_path_literal(BENCHMARKED_FACT_PATH)})
        """
    )
    .df()["column_name"]
    .tolist()
)

parquet_validation = pd.DataFrame([
    {
        "validation_test": "Exported benchmark fact row count matches internal table",
        "passed": exported_fact_row_count == output_fact_row_count,
        "details": f"{exported_fact_row_count} vs {output_fact_row_count}",
    },
    {
        "validation_test": "Exported benchmark fact schema matches internal table",
        "passed": exported_fact_columns == fact_columns,
        "details": "",
    },
])

display(parquet_validation)
assert parquet_validation["passed"].all()

,validation_test,passed,details
0,Exported benchmark fact row count matches internal table,True,2125754 vs 2125754
1,Exported benchmark fact schema matches internal table,True,


## 25. Export Validation and Diagnostic Artifacts

In [26]:
validation_exports = {
    "benchmark_standards.csv": benchmark_standards,
    "benchmark_specification_snapshot.csv": benchmark_specification,
    "benchmark_eligibility.csv": benchmark_eligibility,
    "benchmark_coverage.csv": benchmark_coverage,
    "benchmark_peer_size_summary.csv": benchmark_peer_size_summary,
    "manual_reconciliation.csv": manual_reconciliation,
    "selection_logic_validation.csv": selection_logic_validation,
    "benchmark_validation_results.csv": benchmark_validation_results,
    "parquet_validation.csv": parquet_validation,
}

export_manifest_rows = [
    {
        "file_name": BENCHMARKED_FACT_PATH.name,
        "artifact_type": "Benchmarked fact table",
        "row_count": output_fact_row_count,
        "file_size_mb": round(
            BENCHMARKED_FACT_PATH.stat().st_size / (1024 ** 2), 2
        ),
        "output_path": BENCHMARKED_FACT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    }
]

for file_name, dataframe in validation_exports.items():
    output_path = OUTPUT_DIR / file_name
    dataframe.to_csv(output_path, index=False)

    assert output_path.exists()
    assert output_path.stat().st_size > 0

    export_manifest_rows.append({
        "file_name": file_name,
        "artifact_type": "Validation or diagnostic output",
        "row_count": len(dataframe),
        "file_size_mb": round(output_path.stat().st_size / (1024 ** 2), 4),
        "output_path": output_path.relative_to(PROJECT_ROOT).as_posix(),
    })

export_manifest = pd.DataFrame(export_manifest_rows)
display(export_manifest)

,file_name,artifact_type,row_count,file_size_mb,output_path
0,FactDischarge.parquet,Benchmarked fact table,2125754,48.7400,outputs/peer_benchmarks/tables/FactDischarge.parquet
1,benchmark_standards.csv,Validation or diagnostic output,11,0.0019,outputs/peer_benchmarks/benchmark_standards.csv
2,benchmark_specification_snapshot.csv,Validation or diagnostic output,5,0.0015,outputs/peer_benchmarks/benchmark_specification_snapshot.csv
3,benchmark_eligibility.csv,Validation or diagnostic output,1,0.0002,outputs/peer_benchmarks/benchmark_eligibility.csv
4,benchmark_coverage.csv,Validation or diagnostic output,8,0.0005,outputs/peer_benchmarks/benchmark_coverage.csv
5,benchmark_peer_size_summary.csv,Validation or diagnostic output,4,0.0003,outputs/peer_benchmarks/benchmark_peer_size_summary.csv
6,manual_reconciliation.csv,Validation or diagnostic output,12,0.0010,outputs/peer_benchmarks/manual_reconciliation.csv
7,selection_logic_validation.csv,Validation or diagnostic output,1,0.0004,outputs/peer_benchmarks/selection_logic_validation.csv
8,benchmark_validation_results.csv,Validation or diagnostic output,12,0.0007,outputs/peer_benchmarks/benchmark_validation_results.csv
9,parquet_validation.csv,Validation or diagnostic output,2,0.0002,outputs/peer_benchmarks/parquet_validation.csv


## 26. Generate `docs/peer_benchmarks.md`

In [27]:
def dataframe_to_markdown(dataframe, columns):
    selected = dataframe.loc[:, columns].fillna("").astype(str)

    def escape_value(value):
        return value.replace("|", r"\|").replace("\n", " ")

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"

    rows = [
        "| " + " | ".join(escape_value(value) for value in row) + " |"
        for row in selected.itertuples(index=False, name=None)
    ]

    return "\n".join([header, separator, *rows])


standards_markdown = dataframe_to_markdown(
    benchmark_standards,
    ["standard_id", "decision", "rationale"],
)

coverage_markdown = dataframe_to_markdown(
    benchmark_coverage,
    ["benchmark_metric", "benchmark_level", "benchmark_status", "row_n", "row_pct"],
)

peer_size_markdown = dataframe_to_markdown(
    benchmark_peer_size_summary,
    [
        "benchmark_metric",
        "benchmark_level",
        "benchmarked_row_n",
        "min_peer_n",
        "median_peer_n",
        "p95_peer_n",
        "max_peer_n",
    ],
)

validation_markdown = dataframe_to_markdown(
    benchmark_validation_results,
    ["validation_test", "passed", "details"],
)

peer_benchmark_document = "\n".join([
    "# Peer Benchmark Methodology — Hospital Operations & Cost Efficiency",
    "",
    "## Purpose",
    "",
    "This document describes the implemented leave-one-facility-out descriptive peer benchmarks for length of stay and released estimated cost.",
    "",
    "## Primary Peer Definition",
    "",
    "The primary comparison group is `APR-DRG Code × APR Severity of Illness Code`.",
    "",
    "All eligible records from the focal hospital are removed before calculating that hospital's peer expectation.",
    "",
    "## Fallback Peer Definition",
    "",
    "If fewer than 30 eligible comparison discharges remain at the primary level, the benchmark falls back to `APR-DRG Code`.",
    "",
    "If fewer than 30 eligible comparison discharges remain after fallback, the expected value remains unavailable.",
    "",
    "## Measure Eligibility",
    "",
    "- LOS benchmarking uses records where `is_valid_los = 1`.",
    "- Estimated-cost benchmarking uses records where `is_valid_cost = 1`.",
    "- LOS and cost peer populations are calculated independently.",
    "",
    "## LOS Interpretation",
    "",
    "The LOS expectation uses `los_days_lower_bound`.",
    "",
    "Records released as `120 +` contribute an observable lower bound of 120 days.",
    "",
    "## Implementation Standards",
    "",
    standards_markdown,
    "",
    "## Benchmark Coverage",
    "",
    coverage_markdown,
    "",
    "## Peer Comparison Volumes",
    "",
    peer_size_markdown,
    "",
    "## Validation Results",
    "",
    validation_markdown,
    "",
    "## Important Limitations",
    "",
    "- Peer benchmarks are descriptive operational screening tools.",
    "- They are not machine-learning predictions.",
    "- They are not formal clinical risk-adjustment models.",
    "- They do not establish causality, preventability, or hospital quality.",
    "- APR-DRG and severity do not capture all clinical or structural differences among facilities.",
    "- Estimated costs are released analytical estimates rather than audited hospital operating expenses.",
    "",
    "## Downstream Use",
    "",
    "Power BI will use the benchmarked fact table for actual-to-peer-expected ratios, excess LOS days, estimated-cost comparisons, and related operational investigation measures.",
    "",
    "Future model-predicted LOS must remain separate from these descriptive peer expectations.",
    "",
])

PEER_BENCHMARK_DOCUMENT_PATH = DOCS_DIR / "peer_benchmarks.md"
PEER_BENCHMARK_DOCUMENT_PATH.write_text(
    peer_benchmark_document,
    encoding="utf-8",
)

assert PEER_BENCHMARK_DOCUMENT_PATH.exists()
assert PEER_BENCHMARK_DOCUMENT_PATH.stat().st_size > 0

print(
    "Peer benchmark documentation created:",
    PEER_BENCHMARK_DOCUMENT_PATH.relative_to(PROJECT_ROOT),
)

Peer benchmark documentation created: docs\peer_benchmarks.md


## 27. Finalize Export Manifest

In [28]:
document_manifest_row = pd.DataFrame([
    {
        "file_name": PEER_BENCHMARK_DOCUMENT_PATH.name,
        "artifact_type": "Documentation",
        "row_count": pd.NA,
        "file_size_mb": round(
            PEER_BENCHMARK_DOCUMENT_PATH.stat().st_size / (1024 ** 2), 4
        ),
        "output_path": PEER_BENCHMARK_DOCUMENT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    }
])

final_export_manifest = pd.concat(
    [export_manifest, document_manifest_row],
    ignore_index=True,
)

EXPORT_MANIFEST_PATH = OUTPUT_DIR / "export_manifest.csv"
final_export_manifest.to_csv(EXPORT_MANIFEST_PATH, index=False)

for relative_path in final_export_manifest["output_path"]:
    assert (PROJECT_ROOT / relative_path).exists()

display(final_export_manifest)

,file_name,artifact_type,row_count,file_size_mb,output_path
0,FactDischarge.parquet,Benchmarked fact table,2125754,48.7400,outputs/peer_benchmarks/tables/FactDischarge.parquet
1,benchmark_standards.csv,Validation or diagnostic output,11,0.0019,outputs/peer_benchmarks/benchmark_standards.csv
2,benchmark_specification_snapshot.csv,Validation or diagnostic output,5,0.0015,outputs/peer_benchmarks/benchmark_specification_snapshot.csv
3,benchmark_eligibility.csv,Validation or diagnostic output,1,0.0002,outputs/peer_benchmarks/benchmark_eligibility.csv
4,benchmark_coverage.csv,Validation or diagnostic output,8,0.0005,outputs/peer_benchmarks/benchmark_coverage.csv
5,benchmark_peer_size_summary.csv,Validation or diagnostic output,4,0.0003,outputs/peer_benchmarks/benchmark_peer_size_summary.csv
6,manual_reconciliation.csv,Validation or diagnostic output,12,0.0010,outputs/peer_benchmarks/manual_reconciliation.csv
7,selection_logic_validation.csv,Validation or diagnostic output,1,0.0004,outputs/peer_benchmarks/selection_logic_validation.csv
8,benchmark_validation_results.csv,Validation or diagnostic output,12,0.0007,outputs/peer_benchmarks/benchmark_validation_results.csv
9,parquet_validation.csv,Validation or diagnostic output,2,0.0002,outputs/peer_benchmarks/parquet_validation.csv


## 28. Final Benchmark Profile

In [29]:
final_benchmark_profile = con.sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN peer_expected_los_days IS NOT NULL THEN 1 ELSE 0 END) AS los_benchmarked_n,
        SUM(CASE WHEN los_peer_benchmark_level = 'APR-DRG × Severity' THEN 1 ELSE 0 END) AS los_primary_n,
        SUM(CASE WHEN los_peer_benchmark_level = 'APR-DRG' THEN 1 ELSE 0 END) AS los_fallback_n,
        SUM(CASE WHEN los_peer_benchmark_level = 'Unavailable' THEN 1 ELSE 0 END) AS los_unavailable_n,
        SUM(CASE WHEN peer_expected_estimated_cost IS NOT NULL THEN 1 ELSE 0 END) AS cost_benchmarked_n,
        SUM(CASE WHEN cost_peer_benchmark_level = 'APR-DRG × Severity' THEN 1 ELSE 0 END) AS cost_primary_n,
        SUM(CASE WHEN cost_peer_benchmark_level = 'APR-DRG' THEN 1 ELSE 0 END) AS cost_fallback_n,
        SUM(CASE WHEN cost_peer_benchmark_level = 'Unavailable' THEN 1 ELSE 0 END) AS cost_unavailable_n
    FROM "FactDischargeBenchmarked"
    """
).df()

display(final_benchmark_profile)

,total_rows,los_benchmarked_n,los_primary_n,los_fallback_n,los_unavailable_n,cost_benchmarked_n,cost_primary_n,cost_fallback_n,cost_unavailable_n
0,2125754,2120362.0,2117792.0,2570.0,5392.0,2120362.0,2117792.0,2570.0,5392.0


## 29. Clean Temporary Build Artifacts

In [30]:
con.close()

if WORK_DB_PATH.exists():
    WORK_DB_PATH.unlink()

try:
    WORK_DIR.rmdir()
except OSError:
    pass

print("Temporary DuckDB benchmark artifacts removed.")
print("Peer benchmark outputs retained successfully.")

Temporary DuckDB benchmark artifacts removed.
Peer benchmark outputs retained successfully.


# Final Notebook Summary

## Work Completed

- Loaded and validated committed outputs from Notebooks 02–04.
- Confirmed that the Notebook 04 benchmark columns were still empty.
- Loaded the validated physical star schema from Parquet.
- Recovered hospital, APR-DRG, and severity context from dimension tables.
- Profiled LOS and estimated-cost benchmark eligibility.
- Calculated leave-one-facility-out APR-DRG × severity LOS peer expectations.
- Calculated APR-DRG LOS fallback expectations.
- Calculated leave-one-facility-out APR-DRG × severity estimated-cost peer expectations.
- Calculated APR-DRG estimated-cost fallback expectations.
- Enforced the minimum 30-discharge comparison threshold after excluding the focal hospital.
- Resolved primary, fallback, and unavailable benchmark status independently for LOS and estimated cost.
- Populated all six benchmark columns reserved by Notebook 04.
- Preserved the complete Notebook 04 fact-table grain and all non-benchmark values.
- Validated benchmark-level selection logic.
- Validated direct benchmark calculations using an independent query path.
- Validated benchmarked fact-table schema preservation.
- Exported the benchmarked fact table as compressed Parquet.
- Re-read and reconciled the exported Parquet file.
- Generated benchmark coverage and peer-size diagnostics.
- Generated `docs/peer_benchmarks.md`.
- Preserved Notebook 04 outputs as the immutable pre-benchmark physical layer.

## Key Benchmark Decisions

1. The primary descriptive peer group is `APR-DRG × Severity`.
2. APR-DRG alone is the governed fallback peer group.
3. Every focal hospital is excluded from its own comparison population.
4. At least 30 eligible comparison discharges must remain after facility exclusion.
5. LOS and estimated-cost comparison populations are calculated independently.
6. LOS benchmarks use only records where `is_valid_los = 1`.
7. Estimated-cost benchmarks use only records where `is_valid_cost = 1`.
8. Invalid focal records do not receive a corresponding expected value.
9. Hospital key `0` does not receive a leave-one-facility-out benchmark.
10. `DimService` physical grain does not redefine the analytical benchmark grain.
11. Unavailable expectations remain null rather than being represented as zero.
12. Descriptive peer expectations remain separate from future model-predicted LOS.

## Downstream Use

The benchmarked fact table can support governed measures such as:

- Peer-Expected LOS Days
- LOS Actual-to-Peer-Expected Ratio
- Excess LOS Days — Lower Bound
- Peer-Expected Estimated Cost
- Estimated Cost Actual-to-Peer-Expected Ratio
- Estimated Cost Variance to Peer Expected

## Remaining Deferred Work

- Explicit DAX measure implementation
- Small-cell suppression implementation
- Power BI semantic-model construction
- Power BI relationship verification
- Semantic-model memory and performance testing
- Executive and operational dashboard development
- Predictive expected-LOS modeling
- Model calibration and subgroup validation
- Versioned model scoring
- Multi-year compatibility testing
- Microsoft Fabric implementation
- Peer median LOS context benchmark
- Peer median estimated-cost context benchmark

## Scope Boundary

This notebook calculates descriptive peer expectations. It does not perform machine-learning prediction, formal clinical risk adjustment, causal attribution, hospital-quality grading, or clinical recommendation.